In [ ]:
import importlib.util
import inspect
from pathlib import Path
 
import pandas as pd
 
from asterix_decoder.data_items.data_item import ItemXXX
 
 
# ── Helpers ───────────────────────────────────────────────────────────────────
 
def first_existing_path(candidates: list) -> Path | None:
    """Return the first path in the list that exists on disk."""
    for path in map(Path, candidates):
        if path.exists():
            return path
    return None
 
 
# ==============================================================================
# BLOQ 1 · Load binary file
# Goal: read the raw ASTERIX binary stream from disk into memory.
# ==============================================================================
 
BINARY_PATH = first_existing_path([
    Path("raw_data/datos_asterix_radar.ast"),
])
if BINARY_PATH is None:
    raise FileNotFoundError(
        "No ASTERIX binary file found. Update the path candidates."
    )
 
raw_data: bytes = BINARY_PATH.read_bytes()
 
if len(raw_data) < 3:
    raise ValueError("File is too short to contain a valid ASTERIX header.")
 
print(f"Loaded : {BINARY_PATH}")
print(f"Size   : {len(raw_data):,} bytes")
print(f"First header → CAT={raw_data[0]} | LEN={int.from_bytes(raw_data[1:3], 'big')}")

In [ ]:
# ==============================================================================
# BLOQ 2 · Split binary stream into messages
# Goal: parse the raw bytes into messages_df, one row per ASTERIX message.
#       Each row keeps its record_area (payload without the 3-byte header)
#       so that the FSPEC can be extracted in the next block.
# ==============================================================================
 
def split_messages(raw: bytes) -> pd.DataFrame:
    """Walk the binary stream and return a DataFrame with one row per message."""
    view = memoryview(raw)
    total = len(raw)
    rows, offset, msg_id = [], 0, 1
 
    while offset + 3 <= total:
        cat    = view[offset]
        length = int.from_bytes(view[offset + 1:offset + 3], "big")
 
        if length < 3 or offset + length > total:
            raise ValueError(f"Invalid message at offset {offset}: LEN={length}")
 
        rows.append({
            "message_id":  msg_id,
            "offset":      offset,
            "cat":         cat,
            "length":      length,
            "data_record": bytes(view[offset + 3:offset + length]),
        })
        offset += length
        msg_id += 1
 
    return pd.DataFrame(rows)
 
 
messages_df = split_messages(raw_data)
 
print(f"\nMessages found: {len(messages_df):,}")
print(messages_df["cat"].value_counts().sort_index().rename("count").rename_axis("CAT").to_string())

In [ ]:
# ==============================================================================
# BLOQ 3 · Parse FSPEC → expand to one row per field (message × FRN)
# Goal: turn messages_df into fields_df by reading each message's FSPEC and
#       exploding the active FRNs. This is the skeleton of the final table.
# ==============================================================================
 
_ACTIVE_OFFSETS: tuple[tuple[int, ...], ...] = tuple(
    tuple(i for i in range(7) if (b >> (7 - i)) & 1)
    for b in range(256)
)
# Precompute FX bit too (least-significant bit signals continuation)
_HAS_FX: tuple[bool, ...] = tuple(bool(b & 1) for b in range(256))


def parse_fspec(data_record: bytes) -> tuple[list[int], bytes]:
    frns     : list[int] = []
    frn_base : int       = 1
    cursor   : int       = 0
    n        : int       = len(data_record)

    while cursor < n:
        b = data_record[cursor]
        offsets = _ACTIVE_OFFSETS[b]          # O(1) lookup, no loop
        if offsets:
            base = frn_base                   # avoid repeated attr lookup
            frns += [base + off for off in offsets]
        cursor   += 1
        frn_base += 7
        if not _HAS_FX[b]:                    # O(1) lookup
            break

    return frns, data_record[cursor:]


# ==============================================================================
# Apply: iterate over the raw list — avoids pandas per-row boxing overhead
# ==============================================================================

records = messages_df["data_record"].tolist()        # one allocation, then plain list
parsed  = list(map(parse_fspec, records))            # pure Python map, no pandas tax

messages_df = messages_df.assign(
    frns        = [p[0] for p in parsed],
    data_fields = [p[1] for p in parsed],
)

print(messages_df[["message_id", "cat", "frns", "data_fields"]].head(8).to_string(index=False))

 

In [ ]:

from asterix_decoder.data_tables.uap_tables import uap021_df, uap048_df

def load_uap(path: Path, cat: int) -> pd.DataFrame:
    pass
    """Load and normalize UAP file into: cat, frn, item_id, item_name, length_str."""
    df = pd.read_csv(path) if path.suffix == ".csv" else pd.read_excel(path)

    # Normalize column names
    df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]

    # Accept common variants
    rename_map = {
        "frn": "frn",
        "data_item": "item_id",
        "item_id": "item_id",
        "description": "item_name",
        "length_in_octets": "length_str",
    }
    df = df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns})

    required = {"frn", "item_id", "item_name", "length_str"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing UAP columns in {path}: {missing}")

    frn_str = df["frn"].astype(str).str.strip()
    valid = frn_str.str.fullmatch(r"\d+") & ~df["length_str"].astype(str).str.strip().isin(["-"])
    df = df[valid].copy()

    return pd.DataFrame(
        {
            "cat": cat,
            "frn": frn_str[valid].astype(int).values,
            "item_id": df["item_id"].astype(str).str.strip().values,
            "item_name": df["item_name"].fillna("").astype(str).str.strip().values,
            "length_str": df["length_str"].fillna("").astype(str).str.strip().values,
        }
    )


def discover_item_classes(cats: list[int], base_folder: Path = Path("asterix_decoder/data_items")) -> dict[tuple[int, str], type]:
    """Map (cat, item_id) -> class loaded dynamically from CATxxx folders."""
    class_map: dict[tuple[int, str], type] = {}

    for cat in cats:
        folder = base_folder / f"CAT{cat:03d}"   # CAT021, CAT048, ...
        if not folder.exists():
            continue

        for file in folder.glob("*.py"):
            if file.name == "__init__.py":
                continue

            module_name = f"dyn_cat{cat:03d}_{file.stem}"
            spec = importlib.util.spec_from_file_location(module_name, file)
            if spec is None or spec.loader is None:
                continue

            module = importlib.util.module_from_spec(spec)
            spec.loader.exec_module(module)

            for _, cls in inspect.getmembers(module, inspect.isclass):
                if cls.__module__ == module_name and callable(getattr(cls, "get_item_id", None)):
                    item_id = str(cls.get_item_id()).strip()
                    class_map[(cat, item_id)] = cls

    return class_map


def build_instance(row: pd.Series, class_map: dict[tuple[int, str], type]):
    """Create one decoder instance for one UAP row."""
    cat = int(row["cat"])
    item_id = str(row["item_id"]).strip()
    item_name = str(row["item_name"])
    length_str = str(row["length_str"])

    cls = class_map.get((cat, item_id))
    if cls is None:
        return ItemXXX(item_name=item_name, length_str=length_str, item_id=item_id)

    try:
        return cls(item_name=item_name, length_str=length_str)
    except TypeError:
        return cls(item_name, length_str)


# UAP_SOURCES = {
#     48: Path("asterix_decoder/uap_tables/CAT048.csv"),
#     21: Path("asterix_decoder/uap_tables/CAT021.csv"),
# }

uap_df = pd.concat(
    [uap021_df, uap048_df],
    ignore_index=True,
)

CLASS_MAP = discover_item_classes([21, 48])
uap_df["instance"] = uap_df.apply(lambda r: build_instance(r, CLASS_MAP), axis=1)

print(uap_df[["cat", "frn", "item_id", "instance"]].head(10).to_string(index=False))

In [ ]:
def decode_message(cat: int, frns: list[int], data_fields: bytes, frn_map: dict) -> dict[str, any]:
    """Versión sin pandas para máxima velocidad."""
    final_data = {}
    cursor = 0
    for frn in frns:
        instance = frn_map.get((cat, frn))
        if instance is None:
            raise ValueError(f"FRN {frn} not found")
        delta_cursor, instance_data = instance.decode(data_fields[cursor:])
        if instance_data:
            final_data.update(instance_data)
        cursor += delta_cursor
    return final_data

# Precompila el mapa (cat, frn) → instance
frn_map = dict(zip(uap_df[["cat", "frn"]].apply(tuple, axis=1), uap_df["instance"]))

# Aplica sin pandas overhead
messages_df["data"] = [
    decode_message(r.cat, r.frns, r.data_fields, frn_map)
    for r in messages_df.itertuples(index=False)
]

In [ ]:
from pathlib import Path
import pandas as pd
from asterix_decoder.data_tables.csv_table import CSV_CAT021_COLUMNS, CSV_CAT048_COLUMNS

# ==============================================================================
# BLOQ 6 · Explode decoded data into columns and export
# ==============================================================================

cats_presentes = set(pd.to_numeric(messages_df["cat"], errors="coerce").dropna().astype(int).unique())

if cats_presentes == {48}:
    target_cols = CSV_CAT048_COLUMNS.copy()
elif cats_presentes == {21}:
    target_cols = CSV_CAT021_COLUMNS.copy()
else:
    shared = set(CSV_CAT021_COLUMNS).intersection(CSV_CAT048_COLUMNS)
    target_cols = [c for c in CSV_CAT048_COLUMNS if c in shared]

target_cols = [c for c in target_cols if c != "CAT"]
target_cols = ["CAT"] + target_cols

# --- Build rows from decoded dicts ---
rows = []
for r in messages_df.itertuples(index=False):
    data_dict = r.data if isinstance(r.data, dict) else {}
    row_out = {col: data_dict.get(col, None) for col in target_cols[1:]}
    rows.append(row_out)

final_df = pd.DataFrame(rows, columns=target_cols[1:])

# --- CAT column ---
cat_series = pd.to_numeric(messages_df["cat"], errors="coerce").astype("Int64")
final_df.insert(0, "CAT", cat_series.map(lambda x: f"CAT0{x}" if pd.notna(x) else pd.NA))

# ==============================================================================
# FILTERS
# ==============================================================================

# ── Filter 1 · Bounding box Barcelona TMA ─────────────────────────────────────
LAT_MIN, LAT_MAX = 40.9, 41.7
LON_MIN, LON_MAX = 1.5, 2.6

lat_col = next((c for c in final_df.columns if "LAT" in c.upper()), None)
lon_col = next((c for c in final_df.columns if "LON" in c.upper()), None)

lat = pd.to_numeric(final_df[lat_col], errors="coerce")
lon = pd.to_numeric(final_df[lon_col], errors="coerce")

in_bbox =  lat.isna() | (lat.between(LAT_MIN, LAT_MAX) & lon.between(LON_MIN, LON_MAX))
final_df = final_df[in_bbox].reset_index(drop=True)


# ── Filter 2 · Discard unwanted TYP_020 values (Optional) ────────────────────────────────
# TYP_020_DISCARD = {
#     "No detection",
#     "PSR",
#     "SSR",
#     "SSR + PSR",
# }

# keep_typ = ~final_df["TYP_020"].isin(TYP_020_DISCARD)
# final_df = final_df[keep_typ].reset_index(drop=True)

# Reemplazar strings vacíos por nulos, pero sin convertir todo a texto
final_df = final_df.replace(r"^\s*$", pd.NA, regex=True)


In [ ]:
# ==============================================================================
# EXPORT CSV
# ==============================================================================

output_dir = Path("output")
output_dir.mkdir(exist_ok=True)

csv_path = output_dir / "decoded_messages.csv"

final_df.to_csv(csv_path, index=False, sep=";", decimal=",", na_rep="N/A")

print(f"CATS presentes   : {sorted(cats_presentes)}")
print(f"Columnas finales : {len(final_df.columns)}")
print(f"Shape final_df   : {final_df.shape}")
print(f"✓ CSV  → {csv_path}")